# DuckPD Order-Aware Analytics and Indexing

This notebook uses small local datasets to demonstrate DuckPD's stable snapshot order, deterministic tie handling, lazy `.loc`/`.iloc`, MultiIndex selection, cumulative and window analytics, masked assignment, persistence, and direct file output.

The examples intentionally contrast snapshot-backed inputs with external scans whose row order must be declared explicitly.

In [14]:
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import cast

import pandas as pandas
from pandas.testing import assert_frame_equal

import duckpd
from duckpd.errors import UnorderedOperationError

source = pandas.DataFrame(
    {
        "id": [1, 2, 3, 4, 5, 6],
        "desk": ["rates", "equity", "rates", "credit", "equity", "rates"],
        "symbol": ["UST", "NVDA", "UST", "HYG", "NVDA", "UST"],
        "value": [100.0, 200.0, 100.0, 80.0, 200.0, 120.0],
    }
)
session = duckpd.connect()
frame = session.from_pandas(source)

print(frame)
print(f"Executions after registration: {session.execution_count}")

DuckPD DataFrame
Columns: ['id', 'desk', 'symbol', 'value']
Plan: ScanPlan
Executions after registration: 0


## Stable Snapshot Order and Deterministic Ties

Pandas and Arrow inputs are copied or retained as stable snapshots. DuckPD tracks hidden row identity without exposing a synthetic pandas index, allowing first/last duplicate retention and top-N tie behavior to match pandas lazily.

In [15]:
first = frame.drop_duplicates(subset="symbol", keep="first")
last = frame.drop_duplicates(subset="symbol", keep="last")
top_last = frame.nlargest(1, "value", keep="last")

assert session.execution_count == 0
assert_frame_equal(
    first.collect().reset_index(drop=True),
    source.drop_duplicates(subset="symbol", keep="first").reset_index(drop=True),
)
assert_frame_equal(
    last.collect().reset_index(drop=True),
    source.drop_duplicates(subset="symbol", keep="last").reset_index(drop=True),
)
assert_frame_equal(
    top_last.collect().reset_index(drop=True),
    source.nlargest(1, "value", keep="last").reset_index(drop=True),
)
last.collect()

,id,desk,symbol,value
0,4,credit,HYG,80.0
1,5,equity,NVDA,200.0
2,6,rates,UST,120.0


## Lazy MultiIndex `.loc` and Two-Dimensional `.iloc`

Explicit indexes remain metadata in the lazy plan. MultiIndex exact keys and prefixes compile to predicates, while positional row slices use the stable snapshot order. Row selections remain lazy DuckPD objects until collection.

In [16]:
indexed = frame.set_index(["desk", "symbol"])
exact = cast(duckpd.DataFrame, indexed.loc[("equity", "NVDA")])
rates = cast(duckpd.DataFrame, indexed.loc[("rates",)])
positional = cast(duckpd.DataFrame, frame.iloc[1:4, [0, 2, 3]])

assert_frame_equal(
    exact.collect(),
    source.set_index(["desk", "symbol"]).loc[[("equity", "NVDA")]],
)
assert_frame_equal(
    rates.collect(),
    source.set_index(["desk", "symbol"]).loc[["rates"]],
)
assert_frame_equal(
    positional.collect().reset_index(drop=True),
    source.iloc[1:4, [0, 2, 3]].reset_index(drop=True),
)
exact.collect()

id  value
desk   symbol           
equity NVDA     2  200.0
       NVDA     5  200.0

## Cumulative, Shifted, Ranked, Rolling, and Expanding Analytics

Ordering-sensitive transforms compile into DuckDB window expressions. Snapshot order is stable; external scans require an explicit `order_by` declaration.

In [17]:
analytics = frame.assign(
    cumulative_value=frame["value"].cumsum(),
    previous_value=frame["value"].shift(1),
    value_change=frame["value"].diff(),
    value_pct_change=frame["value"].pct_change(),
    first_tie_rank=frame["value"].rank(method="first", ascending=False),
    rolling_3_mean=frame["value"].rolling(3, min_periods=1).mean(),
    expanding_sum=frame["value"].expanding().sum(),
)

result = analytics.collect()
expected = source.assign(
    cumulative_value=source["value"].cumsum(),
    previous_value=source["value"].shift(1),
    value_change=source["value"].diff(),
    value_pct_change=source["value"].pct_change(),
    first_tie_rank=source["value"].rank(method="first", ascending=False),
    rolling_3_mean=source["value"].rolling(3, min_periods=1).mean(),
    expanding_sum=source["value"].expanding().sum(),
)
assert_frame_equal(result, expected)
result

,id,desk,symbol,value,cumulative_value,previous_value,value_change,value_pct_change,first_tie_rank,rolling_3_mean,expanding_sum
0,1,rates,UST,100.0,100.0,NaN,NaN,NaN,4.0,100.000000,100.0
1,2,equity,NVDA,200.0,300.0,100.0,100.0,1.0,1.0,150.000000,300.0
2,3,rates,UST,100.0,400.0,200.0,-100.0,-0.5,5.0,133.333333,400.0
3,4,credit,HYG,80.0,480.0,100.0,-20.0,-0.2,6.0,126.666667,480.0
4,5,equity,NVDA,200.0,680.0,80.0,120.0,1.5,2.0,126.666667,680.0
5,6,rates,UST,120.0,800.0,200.0,-80.0,-0.4,3.0,133.333333,800.0


## External Scans Require an Honest Ordering Contract

CSV, Parquet, SQL, and table scans do not receive an artificial source order. Positional and window operations reject unordered scans; declaring `order_by` makes the guarantee explicit.

In [18]:
temporary_directory = TemporaryDirectory()
output_directory = Path(temporary_directory.name)
csv_path = output_directory / "events.csv"
source.to_csv(csv_path, index=False)

unordered_csv = session.read_csv(csv_path)
try:
    unordered_csv.iloc[1:3]
except UnorderedOperationError as error:
    print(f"Expected unordered-scan error: {error}")
else:
    raise AssertionError("Unordered external scan unexpectedly allowed .iloc")

ordered_csv = session.read_csv(csv_path, order_by="id")
ordered_slice = cast(duckpd.DataFrame, ordered_csv.iloc[1:3])
assert_frame_equal(
    ordered_slice.collect().reset_index(drop=True),
    source.iloc[1:3].reset_index(drop=True),
)
ordered_slice.collect()

Expected unordered-scan error: Positional .iloc slicing requires a guaranteed row order


,id,desk,symbol,value
0,2,equity,NVDA,200.0
1,3,rates,UST,100.0


## Masked Assignment, Persistence, and Direct Outputs

Masked `.loc` assignment updates the lazy frame handle with a relational `CASE` expression. `persist`, `write_csv`, and `write_parquet` are explicit execution boundaries; no operation silently falls back to pandas.

In [19]:
updated = session.from_pandas(source).set_index("id", drop=False)
updated.loc[updated["value"] < 100, "value"] = 100.0

expected_updated = source.set_index("id", drop=False)
expected_updated.loc[expected_updated["value"] < 100, "value"] = 100.0
assert_frame_equal(updated.collect(), expected_updated)

persisted = analytics.persist("order_analytics")
parquet_path = output_directory / "order_analytics.parquet"
summary_csv_path = output_directory / "order_analytics.csv"
persisted.write_parquet(parquet_path, overwrite=True)
persisted.write_csv(summary_csv_path)

assert parquet_path.exists() and summary_csv_path.exists()
print(f"Session executions: {session.execution_count}")
persisted.head(3)

Session executions: 15


,id,desk,symbol,value,cumulative_value,previous_value,value_change,value_pct_change,first_tie_rank,rolling_3_mean,expanding_sum
0,1,rates,UST,100.0,100.0,NaN,NaN,NaN,4.0,100.000000,100.0
1,2,equity,NVDA,200.0,300.0,100.0,100.0,1.0,1.0,150.000000,300.0
2,3,rates,UST,100.0,400.0,200.0,-100.0,-0.5,5.0,133.333333,400.0


## Takeaways

- Snapshot-backed frames have stable hidden row identity without gaining a synthetic pandas index.
- External scans require explicit ordering for positional and window operations.
- Indexing and analytical transforms remain lazy until collection, persistence, preview, or output.
- Unsupported semantics fail explicitly rather than triggering hidden pandas materialization.